# Notebook 38 — Standalone Adaptive Serving API Benchmark

Load only Hugging Face serving artifacts—checkpoint, preprocessing, and exact catalog—then benchmark the complete request flow: strict-prior events → user encoding → fixed-policy exact retrieval → FastAPI response. No Drive or Gold data is used.

In [ ]:
from google.colab import files, userdata
from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile
WORK=Path('/content/recsys_serving_benchmark'); SOURCE=WORK/'runtime'; SNAPSHOT=WORK/'hf'; REPORT=WORK/'report'
for p in (SOURCE,SNAPSHOT,REPORT): p.mkdir(parents=True,exist_ok=True)
required={'torch':'torch','pandas':'pandas','huggingface_hub':'huggingface_hub>=0.32','fastapi':'fastapi==0.115.0','uvicorn':'uvicorn==0.30.6','httpx':'httpx==0.27.2'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import torch
assert torch.cuda.is_available(),'Select an L4 or A100 GPU.'
free=shutil.disk_usage('/content').free/1024**3; assert free>=12,f'Need at least 12 GiB; found {free:.1f}'
print('GPU/disk preflight:',torch.cuda.get_device_name(0),f'{free:.1f} GiB')

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
REPO='khoa1505/recsys-artifacts'; RUNTIME='runtime/adaptive_serving_benchmark_runtime_v1.zip'; EXPECTED='649882741c920cb5c12e4d4325baaf699d8583d4200c8108a237e57f2cac9430'
def normalize_token(raw):
    if isinstance(raw,bytes): raw=raw.decode(errors='ignore')
    if isinstance(raw,str):
        raw=raw.strip(); return raw if raw.startswith('hf_') else None
    if isinstance(raw,dict):
        for value in raw.values():
            found=normalize_token(value)
            if found: return found
    if isinstance(raw,(list,tuple)):
        for value in raw:
            found=normalize_token(value)
            if found: return found
    return None
try: token=normalize_token(userdata.get('HF_TOKEN'))
except Exception: token=None
token=token or normalize_token(getpass('Paste Hugging Face READ token: '))
assert token,'HF_TOKEN is missing or invalid.'
runtime=Path(hf_hub_download(REPO,RUNTIME,repo_type='model',token=token)); assert hashlib.sha256(runtime.read_bytes()).hexdigest()==EXPECTED
snapshot_download(REPO,repo_type='model',token=token,local_dir=SNAPSHOT,allow_patterns=['v4_corrected/v1/best_model.pt','serving/corrected_v4_2_exact_v1/**','serving/corrected_v4_2_preprocessing_v1/**'])
CATALOG=SNAPSHOT/'serving/corrected_v4_2_exact_v1'; PREPROCESSING=SNAPSHOT/'serving/corrected_v4_2_preprocessing_v1'; CHECKPOINT=SNAPSHOT/'v4_corrected/v1/best_model.pt'
assert (CATALOG/'manifest.json').exists() and (PREPROCESSING/'manifest.json').exists() and CHECKPOINT.stat().st_size>100_000_000
shutil.rmtree(SOURCE,ignore_errors=True); SOURCE.mkdir(parents=True)
with zipfile.ZipFile(runtime) as z: assert z.testzip() is None; z.extractall(SOURCE)
SCRIPT=SOURCE/'scripts/benchmark_adaptive_serving_runtime.py'; assert SCRIPT.exists(); compile(SCRIPT.read_text(),str(SCRIPT),'exec')
print('all standalone artifacts/runtime: PASS')

In [ ]:
command=[sys.executable,str(SCRIPT),'--catalog-dir',str(CATALOG),'--preprocessing-dir',str(PREPROCESSING),'--checkpoint',str(CHECKPOINT),'--output-dir',str(REPORT),'--iterations','20','--batch-sizes','1,8,32','--top-k','10','--device','cuda']
log=Path('/content/adaptive_serving_benchmark.log'); process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with log.open('w') as stream:
    for line in process.stdout: print(line,end=''); stream.write(line); stream.flush()
code=process.wait()
if code!=0: raise RuntimeError('Serving benchmark failed:\n'+'\n'.join(log.read_text(errors='replace').splitlines()[-240:]))
report=json.loads((REPORT/'serving_benchmark.json').read_text())
assert report['gold_data_read'] is False and all(report['api_smoke'].values())
print(json.dumps(report,indent=2))

In [ ]:
small=WORK/'download'; shutil.rmtree(small,ignore_errors=True); small.mkdir(); shutil.copy2(REPORT/'serving_benchmark.json',small/'serving_benchmark.json'); shutil.copy2(log,small/log.name)
archive=Path(shutil.make_archive('/content/adaptive_serving_api_benchmark_report','zip',small))
print('download:',archive); files.download(str(archive))